# Notebook 13: HuggingFace Datasets

**Learning Objectives:**
- Load and explore datasets from the HuggingFace Hub
- Filter, map, and preprocess datasets efficiently
- Create custom datasets from local files
- Understand dataset splits, shuffling, and sampling
- Integrate datasets with PyTorch DataLoaders for training

**Prerequisites:** Notebooks 01–03 (basic HuggingFace familiarity)

## Prerequisites

### Hardware Requirements

| Requirement | Specification |
|-------------|---------------|
| **CPU** | Any modern CPU |
| **RAM** | 4GB minimum |
| **Storage** | 2GB free (for downloaded datasets) |
| **GPU** | Not required |

### Software Requirements
- Python 3.8+
- Libraries: `datasets`, `transformers`, `torch`, `pandas`
- See `requirements.txt` for full list

## Expected Behaviors

### First Time Running
- Datasets download from HuggingFace Hub on first use
- `imdb` dataset: ~85MB download, ~130MB cached
- `ag_news` dataset: ~30MB download
- Subsequent runs use the cached version (instant)

### Dataset Loading
```
Downloading and preparing dataset imdb/plain_text...
Dataset imdb downloaded and prepared.
```

### Output Format
- Datasets display as formatted tables with column names and types
- Individual examples display as dictionaries
- Mapping operations show a progress bar

### Common Issues
- **Slow download**: Large datasets may take minutes on slow connections
- **Disk space**: Cached datasets live in `~/.cache/huggingface/datasets/`
- **Memory**: Very large datasets use memory-mapped files (efficient)

## Overview

The **HuggingFace Datasets** library provides a unified interface for loading, processing, and sharing datasets. It supports thousands of community-contributed datasets across NLP, computer vision, and audio.

**Key Features:**
- **Hub integration**: Load any of 100,000+ datasets with a single line
- **Memory efficiency**: Uses Apache Arrow for memory-mapped storage
- **Fast processing**: `.map()` and `.filter()` with multiprocessing support
- **Interoperability**: Works seamlessly with PyTorch, TensorFlow, and pandas

**Why This Matters:**
- Data quality determines model quality — garbage in, garbage out
- Proper preprocessing pipelines prevent subtle bugs
- Understanding your data helps you choose the right model and detect biases

## Setup and Installation

In [ ]:
# Import libraries
import random
import sys
import numpy as np
import pandas as pd
import torch
import transformers
import datasets as datasets_lib
from datasets import (
    load_dataset,
    Dataset,
    DatasetDict,
    Features,
    ClassLabel,
    Value,
)
from transformers import AutoTokenizer
import warnings
warnings.filterwarnings('ignore')

# Set seed for reproducibility
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device selection
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Version metadata
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"Datasets: {datasets_lib.__version__}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Part 1: Loading Datasets from the Hub

The `load_dataset()` function is the main entry point for the datasets library. It can load datasets from the HuggingFace Hub, from local files, or from in-memory data. Each dataset is returned as a `DatasetDict` containing named splits (train, test, validation).

In [ ]:
# Load a popular NLP dataset
imdb = load_dataset("imdb")

print("=== IMDB DATASET ===")
print(f"Type: {type(imdb)}")
print(f"Splits: {list(imdb.keys())}")
print(f"\nTraining set size: {len(imdb['train']):,}")
print(f"Test set size: {len(imdb['test']):,}")
print(f"\nFeatures: {imdb['train'].features}")
print(f"\nFirst example:")
print(f"  Text: {imdb['train'][0]['text'][:100]}...")
print(f"  Label: {imdb['train'][0]['label']} (0=negative, 1=positive)")

### Loading Specific Splits and Subsets

You don't always need the full dataset. The `split` parameter lets you load specific splits, and you can even slice them with string indexing. This is useful for quick prototyping or when working with limited memory.

In [ ]:
# Load only the first 1000 training examples
imdb_small = load_dataset("imdb", split="train[:1000]")
print(f"Small subset: {len(imdb_small)} examples")

# Load a percentage
imdb_10pct = load_dataset("imdb", split="train[:10%]")
print(f"10% subset: {len(imdb_10pct)} examples")

# Load multiple splits at once
train_and_test = load_dataset("imdb", split=["train", "test"])
print(f"\nLoaded {len(train_and_test)} splits: {[len(s) for s in train_and_test]} examples each")

### Exploring Dataset Metadata

Every dataset on the Hub comes with metadata describing its features, size, license, and citation information. The `info` attribute provides this metadata programmatically, which is useful for automated data quality checks.

In [ ]:
# Explore dataset metadata
print("=== DATASET INFO ===")
print(f"Description: {imdb['train'].info.description[:200]}...")
print(f"\nHomepage: {imdb['train'].info.homepage}")
print(f"License: {imdb['train'].info.license}")
print(f"\nFeatures:")
for name, feature in imdb['train'].features.items():
    print(f"  {name}: {feature}")

# Dataset size on disk
print(f"\nDataset size: {imdb['train'].info.dataset_size / (1024**2):.1f} MB" if imdb['train'].info.dataset_size else "")

## Part 2: Exploring and Filtering Data

Before using a dataset for training, you should explore its contents to understand the distribution of labels, text lengths, and potential quality issues. The datasets library provides efficient methods for filtering and selecting subsets without loading everything into memory.

In [ ]:
# Basic statistics
train_data = imdb['train']

# Label distribution
labels = train_data['label']
label_counts = {0: labels.count(0), 1: labels.count(1)}

print("=== LABEL DISTRIBUTION ===")
label_df = pd.DataFrame({
    "Label": ["Negative (0)", "Positive (1)"],
    "Count": [label_counts[0], label_counts[1]],
    "Percentage": [f"{label_counts[0]/len(labels):.1%}", f"{label_counts[1]/len(labels):.1%}"]
})
print(label_df.to_string(index=False))

# Text length distribution
text_lengths = [len(text.split()) for text in train_data['text'][:1000]]
print(f"\n=== TEXT LENGTH (first 1000 examples) ===")
print(f"Mean: {np.mean(text_lengths):.0f} words")
print(f"Median: {np.median(text_lengths):.0f} words")
print(f"Min: {np.min(text_lengths)} words")
print(f"Max: {np.max(text_lengths)} words")

### Filtering Datasets

The `.filter()` method applies a boolean function to each example and keeps only those that return `True`. This is useful for removing low-quality examples, selecting specific categories, or creating balanced subsets.

In [ ]:
# Filter examples by criteria
def is_long_review(example: dict) -> bool:
    """Check if a review has more than 200 words.

    Args:
        example: A single dataset example with a 'text' field.

    Returns:
        True if the text has more than 200 words.
    """
    return len(example['text'].split()) > 200


# Filter for long reviews only
long_reviews = train_data.filter(is_long_review)
print(f"Original size: {len(train_data):,}")
print(f"Long reviews (>200 words): {len(long_reviews):,}")
print(f"Filtered out: {len(train_data) - len(long_reviews):,} short reviews")

# Filter for positive reviews only
positive_reviews = train_data.filter(lambda x: x['label'] == 1)
print(f"\nPositive reviews: {len(positive_reviews):,}")

### Selecting and Slicing

For quick inspection, `.select()` lets you pick specific indices, and standard Python slicing works on the underlying data. Converting to pandas DataFrames is also useful for familiar exploratory analysis.

In [ ]:
# Select specific examples by index
sample = train_data.select(range(5))
print("=== FIRST 5 EXAMPLES ===")
for i, example in enumerate(sample):
    label_str = "Positive" if example['label'] == 1 else "Negative"
    print(f"{i+1}. [{label_str}] {example['text'][:80]}...")

# Convert to pandas for exploration
print("\n=== AS PANDAS DATAFRAME ===")
df = train_data.select(range(100)).to_pandas()
print(df.head())
print(f"\nShape: {df.shape}")
print(f"Dtypes:\n{df.dtypes}")

## Part 3: Preprocessing and Transformations

The `.map()` function is the workhorse of dataset preprocessing. It applies a transformation function to every example (or batch of examples) and can add, modify, or remove columns. Crucially, `.map()` caches its results — so re-running the same transformation is instant.

In [ ]:
# Add a new column: word count
def add_word_count(example: dict) -> dict:
    """Add word_count field to a dataset example.

    Args:
        example: A single dataset example with a 'text' field.

    Returns:
        The example with an added 'word_count' field.
    """
    example['word_count'] = len(example['text'].split())
    return example


# Apply to first 5000 examples for speed
subset = train_data.select(range(5000))
subset_with_counts = subset.map(add_word_count)

print("=== WORD COUNT STATISTICS ===")
print(f"Columns: {subset_with_counts.column_names}")
counts = subset_with_counts['word_count']
print(f"Mean: {np.mean(counts):.0f} words")
print(f"Std: {np.std(counts):.0f} words")

### Tokenization with Map

The most common preprocessing step for NLP is tokenization. By combining `.map()` with a HuggingFace tokenizer, you can efficiently tokenize an entire dataset. Using `batched=True` processes multiple examples at once, which is significantly faster for large datasets.

In [ ]:
# Tokenize the dataset
TOKENIZER_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)


def tokenize_function(examples: dict) -> dict:
    """Tokenize a batch of text examples.

    Args:
        examples: Batch of examples with 'text' field.

    Returns:
        Tokenized batch with input_ids and attention_mask.
    """
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=128,
    )


# Tokenize with batched processing (much faster)
print("Tokenizing dataset (batched)...")
tokenized_subset = subset.map(tokenize_function, batched=True, batch_size=256)

print(f"\n=== TOKENIZED DATASET ===")
print(f"Columns: {tokenized_subset.column_names}")
print(f"\nFirst example tokens (first 20):")
print(f"  input_ids: {tokenized_subset[0]['input_ids'][:20]}")
print(f"  attention_mask: {tokenized_subset[0]['attention_mask'][:20]}")
print(f"  Decoded: {tokenizer.decode(tokenized_subset[0]['input_ids'][:20])}")

### Removing and Renaming Columns

After tokenization, you may want to clean up the dataset by removing the raw text column (which the model doesn't need) or renaming columns to match what the model expects. These operations are lightweight since they only modify metadata, not the underlying data.

In [ ]:
# Remove raw text column (model doesn't need it)
model_ready = tokenized_subset.remove_columns(['text'])
print(f"Columns after removing 'text': {model_ready.column_names}")

# Rename 'label' to 'labels' (what most HuggingFace models expect)
model_ready = model_ready.rename_column('label', 'labels')
print(f"Columns after rename: {model_ready.column_names}")

# Set format for PyTorch
model_ready.set_format('torch')
print(f"\nFormat: {model_ready.format}")
print(f"First example type: {type(model_ready[0]['input_ids'])}")
print(f"Shape: {model_ready[0]['input_ids'].shape}")

## Part 4: Creating Custom Datasets

You won't always use pre-built datasets from the Hub. The datasets library makes it easy to create datasets from dictionaries, pandas DataFrames, CSV files, and JSON files. This is essential when working with your own data.

In [ ]:
# Create from a Python dictionary
custom_data = {
    "text": [
        "The product works great!",
        "Terrible quality, broke after one day.",
        "Average product, nothing special.",
        "Best purchase I've ever made!",
        "Would not recommend to anyone.",
        "Decent for the price.",
        "Exceeded my expectations completely.",
        "Waste of money, very disappointed.",
    ],
    "label": [1, 0, 1, 1, 0, 1, 1, 0],
    "rating": [5, 1, 3, 5, 1, 3, 5, 1],
}

custom_dataset = Dataset.from_dict(custom_data)

print("=== CUSTOM DATASET ===")
print(f"Size: {len(custom_dataset)}")
print(f"Features: {custom_dataset.features}")
print(f"Columns: {custom_dataset.column_names}")
print(f"\nAs DataFrame:")
print(custom_dataset.to_pandas().to_string(index=False))

In [ ]:
# Create from a pandas DataFrame
df = pd.DataFrame({
    "sentence": [
        "Machine learning is fascinating.",
        "Deep learning requires lots of data.",
        "NLP has made great progress recently.",
        "Computer vision uses convolutional networks.",
    ],
    "category": ["general", "general", "nlp", "cv"],
})

df_dataset = Dataset.from_pandas(df)
print("=== FROM PANDAS ===")
print(f"Size: {len(df_dataset)}")
print(f"Features: {df_dataset.features}")
print(df_dataset.to_pandas().to_string(index=False))

### Creating Train/Test Splits

When you create a custom dataset, you'll typically need to split it into training and test sets. The `.train_test_split()` method handles this with stratified sampling support to maintain label proportions across splits.

In [ ]:
# Split custom dataset into train/test
split_dataset = custom_dataset.train_test_split(test_size=0.25, seed=SEED)

print("=== TRAIN/TEST SPLIT ===")
print(f"Type: {type(split_dataset)}")
print(f"Train size: {len(split_dataset['train'])}")
print(f"Test size: {len(split_dataset['test'])}")

# Check label distribution in both splits
for split_name in ['train', 'test']:
    labels = split_dataset[split_name]['label']
    pos = sum(labels)
    neg = len(labels) - pos
    print(f"\n{split_name}: {pos} positive, {neg} negative ({pos/len(labels):.0%} positive)")

## Part 5: Shuffling, Sorting, and Sampling

Proper data shuffling is critical for training — models can learn spurious patterns if examples are ordered by label or source. The datasets library provides efficient shuffling, sorting, and sampling that work even on datasets too large to fit in memory.

In [ ]:
# Shuffling
shuffled = train_data.shuffle(seed=SEED)
print("=== SHUFFLED DATASET ===")
print(f"First 5 labels (original): {train_data['label'][:5]}")
print(f"First 5 labels (shuffled): {shuffled['label'][:5]}")

# Sorting
sorted_by_label = subset_with_counts.sort('word_count')
print(f"\n=== SORTED BY WORD COUNT ===")
print(f"Shortest: {sorted_by_label[0]['word_count']} words")
print(f"Longest: {sorted_by_label[-1]['word_count']} words")

# Random sampling
sample_indices = random.sample(range(len(train_data)), 5)
sample = train_data.select(sample_indices)
print(f"\n=== RANDOM SAMPLE (5 examples) ===")
for i, example in enumerate(sample):
    label_str = "Pos" if example['label'] == 1 else "Neg"
    print(f"  {i+1}. [{label_str}] {example['text'][:60]}...")

## Part 6: Integration with PyTorch DataLoaders

For training, you need to feed batches of data to your model. The datasets library integrates directly with PyTorch's `DataLoader`. After setting the format to `'torch'`, each example is returned as a dictionary of tensors, ready for the model's `forward()` method.

In [ ]:
from torch.utils.data import DataLoader

# Prepare the tokenized dataset for PyTorch
torch_dataset = tokenized_subset.remove_columns(['text'])
torch_dataset = torch_dataset.rename_column('label', 'labels')
torch_dataset.set_format('torch')

# Create DataLoader
BATCH_SIZE = 16
dataloader = DataLoader(torch_dataset, batch_size=BATCH_SIZE, shuffle=True)

# Inspect one batch
batch = next(iter(dataloader))

print("=== DATALOADER BATCH ===")
print(f"Batch size: {BATCH_SIZE}")
print(f"Number of batches: {len(dataloader)}")
print(f"\nBatch keys: {list(batch.keys())}")
for key, tensor in batch.items():
    print(f"  {key}: shape={tensor.shape}, dtype={tensor.dtype}")

In [ ]:
# Example: iterate through batches (simulating a training loop)
print("=== SIMULATED TRAINING LOOP ===")
print(f"Total examples: {len(torch_dataset)}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Batches per epoch: {len(dataloader)}")

for i, batch in enumerate(dataloader):
    input_ids = batch['input_ids']
    labels = batch['labels']

    if i < 3:  # Show first 3 batches
        print(f"\nBatch {i+1}:")
        print(f"  input_ids shape: {input_ids.shape}")
        print(f"  labels: {labels.tolist()}")

    if i >= 2:
        remaining = len(dataloader) - 3
        print(f"\n... {remaining} more batches")
        break

## Practical Applications

Let's put everything together with two real-world examples: loading a multi-class classification dataset and combining multiple datasets into one.

In [ ]:
# Example 1: Load and explore AG News (4-class text classification)
ag_news = load_dataset("ag_news", split="train[:2000]")

label_names = ["World", "Sports", "Business", "Sci/Tech"]

print("=== AG NEWS DATASET ===")
print(f"Size: {len(ag_news)}")
print(f"Features: {ag_news.features}")

# Label distribution
label_counts = {}
for label in ag_news['label']:
    label_counts[label] = label_counts.get(label, 0) + 1

dist_df = pd.DataFrame({
    "Category": [label_names[i] for i in sorted(label_counts.keys())],
    "Count": [label_counts[i] for i in sorted(label_counts.keys())],
})
dist_df["Percentage"] = (dist_df["Count"] / dist_df["Count"].sum() * 100).round(1).astype(str) + "%"
print(f"\n{dist_df.to_string(index=False)}")

# Show one example per category
print("\n=== EXAMPLE PER CATEGORY ===")
for label_id, name in enumerate(label_names):
    example = ag_news.filter(lambda x: x['label'] == label_id).select([0])[0]
    print(f"\n[{name}] {example['text'][:100]}...")

In [ ]:
# Example 2: Combine datasets with interleaving
from datasets import concatenate_datasets

# Create two small datasets
positive_data = Dataset.from_dict({
    "text": ["Great movie!", "Loved it!", "Excellent film!"],
    "label": [1, 1, 1],
    "source": ["dataset_a", "dataset_a", "dataset_a"],
})

negative_data = Dataset.from_dict({
    "text": ["Terrible movie.", "Hated it.", "Awful film."],
    "label": [0, 0, 0],
    "source": ["dataset_b", "dataset_b", "dataset_b"],
})

# Concatenate
combined = concatenate_datasets([positive_data, negative_data])
combined = combined.shuffle(seed=SEED)

print("=== COMBINED DATASET ===")
print(combined.to_pandas().to_string(index=False))

## Performance Benchmarking

The datasets library uses Apache Arrow under the hood, which provides memory-mapped storage and fast columnar access. Let's measure how different operations perform and understand the speed benefits of batched processing.

In [ ]:
import time

# Benchmark: map with vs without batching
benchmark_data = load_dataset("imdb", split="train[:5000]")


def simple_transform(example: dict) -> dict:
    """Add uppercase text field.

    Args:
        example: Dataset example with 'text' field.

    Returns:
        Example with added 'text_upper' field.
    """
    example['text_upper'] = example['text'].upper()
    return example


def batched_transform(examples: dict) -> dict:
    """Add uppercase text field (batched version).

    Args:
        examples: Batch of examples with 'text' field.

    Returns:
        Batch with added 'text_upper' field.
    """
    examples['text_upper'] = [t.upper() for t in examples['text']]
    return examples


# Single-example processing
start = time.perf_counter()
_ = benchmark_data.map(simple_transform)
single_time = time.perf_counter() - start

# Batched processing
start = time.perf_counter()
_ = benchmark_data.map(batched_transform, batched=True, batch_size=256)
batched_time = time.perf_counter() - start

print("=== MAP PERFORMANCE ===")
results_df = pd.DataFrame({
    "Method": ["Single example", "Batched (256)"],
    "Time (s)": [f"{single_time:.2f}", f"{batched_time:.2f}"],
    "Speedup": ["1.0x (baseline)", f"{single_time/batched_time:.1f}x"],
})
print(results_df.to_string(index=False))

## Exercises

1. **Dataset Exploration**: Load the `rotten_tomatoes` dataset. Compare its size, label distribution, and average text length with IMDB.

2. **Custom Preprocessing**: Write a `.map()` function that lowercases text, removes punctuation, and adds a `clean_text` column to the IMDB dataset.

3. **Stratified Sampling**: Create a balanced subset of 100 positive and 100 negative reviews from the IMDB training set.

4. **Cross-Dataset Merge**: Load both `imdb` and `rotten_tomatoes` datasets, unify their column names, and concatenate them into a single dataset.

5. **DataLoader Experiment**: Create a DataLoader with different batch sizes (8, 32, 128) and measure how batch size affects iteration speed.

In [ ]:
# Try the exercises above to deepen your understanding of HuggingFace Datasets.
# For example, load rotten_tomatoes:
#   rt = load_dataset("rotten_tomatoes")
#   print(rt)

## Key Takeaways

- **`load_dataset()`** provides one-line access to 100,000+ datasets on the Hub
- **`.filter()` and `.select()`** let you create subsets without copying data
- **`.map()`** with `batched=True` is the fastest way to preprocess datasets
- **Custom datasets** can be created from dicts, DataFrames, CSV, or JSON
- **`.train_test_split()`** handles splitting with optional stratification
- **`.set_format('torch')`** makes datasets ready for PyTorch DataLoaders

## Next Steps

- Try **Notebook 14**: Gradio and HuggingFace Spaces
- Explore the [HuggingFace Datasets documentation](https://huggingface.co/docs/datasets/)
- Browse available datasets at [huggingface.co/datasets](https://huggingface.co/datasets)

## Resources

- [Datasets Documentation](https://huggingface.co/docs/datasets/)
- [Dataset Loading Guide](https://huggingface.co/docs/datasets/loading)
- [Processing Guide](https://huggingface.co/docs/datasets/process)
- [Apache Arrow Format](https://arrow.apache.org/)